# Sales Prediction using Machine Learning

## Objective

**What is the problem?**
Predicting the amount of sales based on advertising investments in TV, Radio, and Newspaper channels.

**Why is it useful?**
It allows businesses to optimize their marketing budgets by allocating funds to the highest-ROI channels.

**Expected outcome.**
A regression model that accurately forecasts sales, alongside insights into feature importance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import warnings
warnings.filterwarnings('ignore')

## Dataset Loading & Statistics

In [ ]:
dataset_path = '../dataset/Advertising.csv'
if not os.path.exists(dataset_path):
    print('Please place the Advertising dataset as Advertising.csv in the dataset/ folder.')
else:
    df = pd.read_csv(dataset_path)
    if 'Unnamed: 0' in df.columns:
        df.drop('Unnamed: 0', axis=1, inplace=True)
    display(df.head())
    display(df.info())
    print('Null values:\n', df.isnull().sum())
    display(df.describe())

## Exploratory Data Analysis

In [ ]:
if os.path.exists(dataset_path):
    sns.pairplot(df)
    plt.savefig('../images/pairplot.png')
    plt.show()

### Observations
- TV advertising shows a strong linear relationship with Sales.
- Radio has a moderate positive correlation.
- Newspaper shows a very weak and noisy relationship with Sales.

### Scatter Plots (Medium vs Sales)

In [ ]:
if os.path.exists(dataset_path):
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    sns.scatterplot(data=df, x='TV', y='Sales', ax=axes[0], color='blue')
    axes[0].set_title('TV vs Sales')
    sns.scatterplot(data=df, x='Radio', y='Sales', ax=axes[1], color='green')
    axes[1].set_title('Radio vs Sales')
    sns.scatterplot(data=df, x='Newspaper', y='Sales', ax=axes[2], color='red')
    axes[2].set_title('Newspaper vs Sales')
    plt.tight_layout()
    plt.savefig('../images/scatter_plots.png')
    plt.show()

### Observations
- The scatter plots visually confirm that TV budgets are the safest driver of Sales.

### Correlation Heatmap

In [ ]:
if os.path.exists(dataset_path):
    plt.figure(figsize=(6, 4))
    sns.heatmap(df.corr(), annot=True, cmap='YlGnBu')
    plt.title('Correlation Heatmap')
    plt.savefig('../images/correlation_heatmap.png')
    plt.show()

### Observations
- TV and Sales: 0.78 (Very Strong).
- Radio and Sales: 0.58 (Moderate).
- Newspaper and Sales: 0.23 (Weak).

## Train Test Split

In [ ]:
if os.path.exists(dataset_path):
    X = df.drop('Sales', axis=1)
    y = df['Sales']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Model Training
We train Linear Regression, Random Forest, and a Polynomial Regression model (degree 2) to capture potential synergy between TV and Radio advertising.

In [ ]:
if os.path.exists(dataset_path):
    models = {
        'Linear Regression': LinearRegression(),
        'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
        'Polynomial Regression (d=2)': make_pipeline(PolynomialFeatures(degree=2), LinearRegression())
    }
    results = {}

## Model Evaluation

In [ ]:
if os.path.exists(dataset_path):
    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = r2_score(y_test, y_pred)
        results[name] = r2
        
        print(f'--- {name} ---')
        print(f'MAE: {mae:.2f}')
        print(f'RMSE: {rmse:.2f}')
        print(f'R2 Score: {r2:.2f}\n')

### Metric Explanations
- **MAE**: Average error in predicted sales.
- **RMSE**: Heavily penalizes large prediction errors.
- **R² Score**: How well the variance in Sales is explained by the model.

### Residual Plot for Best Model

In [ ]:
if os.path.exists(dataset_path):
    best_model_name = max(results, key=results.get)
    best_model = models[best_model_name]
    y_pred = best_model.predict(X_test)
    residuals = y_test - y_pred
    
    plt.figure(figsize=(8, 5))
    sns.scatterplot(x=y_pred, y=residuals, color='purple')
    plt.axhline(y=0, color='r', linestyle='--')
    plt.title('Residual Plot')
    plt.xlabel('Predicted Sales')
    plt.ylabel('Residuals')
    plt.savefig('../images/residual_plot.png')
    plt.show()

### Interpretations
- **Which advertising medium contributes most?** TV is the dominant contributor.
- Interestingly, Polynomial Regression usually performs exceptionally well here because there is a known synergy effect (interaction) between TV and Radio ads in this specific dataset.

## Support Conclusion using Regression Coefficients

In [ ]:
if os.path.exists(dataset_path):
    lr = models['Linear Regression']
    coeffs = pd.DataFrame({'Feature': X.columns, 'Coefficient': lr.coef_})
    coeffs.sort_values(by='Coefficient', ascending=False, inplace=True)
    
    plt.figure(figsize=(6, 4))
    sns.barplot(data=coeffs, x='Feature', y='Coefficient', palette='magma')
    plt.title('Linear Regression Coefficients')
    plt.show()
    display(coeffs)

## Save Best Model

In [ ]:
if os.path.exists(dataset_path):
    joblib.dump(best_model, '../models/best_sales_model.pkl')
    print(f'Best model ({best_model_name}) saved.')

## Final Conclusion
- **Insights**: To maximize ROI, the marketing budget should heavily favor TV, followed by Radio. Newspaper advertising does not yield significant returns.
- **Model Performance**: Polynomial and Random Forest models capture the non-linear synergy between TV and Radio, achieving excellent R2 scores.
- **Future Improvements**: Using more granular data (e.g., daily sales, digital ad spend) could provide a more holistic view.